# 02. Experiments and model selection

## Цель

В этом notebook сравниваются ключевые подходы к предсказанию цены автомобиля на **одном фиксированном holdout-разбиении**. Он отвечает на вопрос: какие компоненты были полезны и почему финальное решение является ансамблем.

Проверяемые компоненты:

1. `Ridge` на табличных признаках — линейный baseline;
2. `Text Ridge` — текстовая модель по названию и описанию автомобиля;
3. `CatBoost V8` — основная нелинейная модель на технических и извлечённых текстовых признаках;
4. `CatBoost V10` — V8, дополненная fold-safe target statistics;
5. `Retrieval K=3` — оценка цены по ближайшим аналогам.

> **Важно:** этот notebook предназначен для интерпретируемого сравнения подходов. Окончательные веса ансамбля и calibration не выбираются по этому holdout: они были ранее зафиксированы по 5-fold OOF и nested validation. Обучение всех утверждённых моделей на всей выборке будет выполнено в `03_final_model_training.ipynb`.

## 1. Импорты, пути и воспроизводимость

Notebook использует только подготовленные данные из `01_data_eda_features.ipynb`. Feature engineering здесь не повторяется: это исключает расхождение между экспериментами и финальным обучением.

In [4]:
# ==============================================================
# 1. PROJECT SETUP
# ============================================================== 

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
VALID_SIZE = 0.20

current_path = Path.cwd().resolve()
possible_roots = [current_path, *current_path.parents]

PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "data").exists()
        and (path / "train_data").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта.\n"
        f"Текущая папка: {current_path}\n\n"
        "Ожидалась структура shift_ml/data и shift_ml/train_data."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Reports:", REPORTS_DIR)

Project root: C:\temp\shift_ml
Processed data: C:\temp\shift_ml\data\processed
Reports: C:\temp\shift_ml\reports


## 2. Загрузка подготовленных данных

Загружаем сохранённые V8-признаки, а также canonical-таблицы. Canonical-таблицы используются только для текстового компонента: в V8 feature set исходное поле `Полное название` намеренно заменено более устойчивыми производными признаками.

In [5]:
# ==============================================================
# 2. LOAD PREPARED DATA
# ============================================================== 

TRAIN_MODEL_INPUT_PATH = PROCESSED_DIR / "train_model_input_v8.parquet"
TEST_MODEL_INPUT_PATH = PROCESSED_DIR / "test_model_input_v8.parquet"
X_TRAIN_V8_PATH = PROCESSED_DIR / "X_train_v8.parquet"
X_TEST_V8_PATH = PROCESSED_DIR / "X_test_v8.parquet"
TRAIN_CANONICAL_PATH = PROCESSED_DIR / "train_canonical.parquet"
TEST_CANONICAL_PATH = PROCESSED_DIR / "X_test_canonical.parquet"
FEATURE_SCHEMA_PATH = PROCESSED_DIR / "feature_schema.json"

required_paths = {
    "train_model_input_v8.parquet": TRAIN_MODEL_INPUT_PATH,
    "test_model_input_v8.parquet": TEST_MODEL_INPUT_PATH,
    "X_train_v8.parquet": X_TRAIN_V8_PATH,
    "X_test_v8.parquet": X_TEST_V8_PATH,
    "train_canonical.parquet": TRAIN_CANONICAL_PATH,
    "X_test_canonical.parquet": TEST_CANONICAL_PATH,
    "feature_schema.json": FEATURE_SCHEMA_PATH,
}

missing_files = {
    name: path
    for name, path in required_paths.items()
    if not path.exists()
}

if missing_files:
    missing_text = "\n".join(
        f"— {name}: {path}"
        for name, path in missing_files.items()
    )
    raise FileNotFoundError(
        "Не найдены артефакты из 01_data_eda_features.ipynb.\n\n"
        f"{missing_text}"
    )

train_model_input = pd.read_parquet(TRAIN_MODEL_INPUT_PATH)
test_model_input = pd.read_parquet(TEST_MODEL_INPUT_PATH)
X_train_v8 = pd.read_parquet(X_TRAIN_V8_PATH)
X_test_v8 = pd.read_parquet(X_TEST_V8_PATH)
train_canonical = pd.read_parquet(TRAIN_CANONICAL_PATH)
test_canonical = pd.read_parquet(TEST_CANONICAL_PATH)

with open(FEATURE_SCHEMA_PATH, encoding="utf-8") as file:
    feature_schema = json.load(file)

TARGET_COLUMN = "Цена"
ID_COLUMN = "car_id"

if list(X_train_v8.columns) != list(X_test_v8.columns):
    raise ValueError("V8 feature columns train/test не совпадают.")

if len(train_model_input) != len(X_train_v8):
    raise ValueError("Размер train_model_input и X_train_v8 не совпадает.")

if len(test_model_input) != len(X_test_v8):
    raise ValueError("Размер test_model_input и X_test_v8 не совпадает.")

if TARGET_COLUMN not in train_model_input.columns:
    raise KeyError(f"В train_model_input нет колонки {TARGET_COLUMN}.")

if ID_COLUMN not in train_model_input.columns:
    raise KeyError(f"В train_model_input нет колонки {ID_COLUMN}.")

y = train_model_input[TARGET_COLUMN].astype(float).reset_index(drop=True)
train_ids = train_model_input[ID_COLUMN].astype(str).reset_index(drop=True)
test_ids = test_model_input[ID_COLUMN].astype(str).reset_index(drop=True)

X_train_v8 = X_train_v8.reset_index(drop=True)
X_test_v8 = X_test_v8.reset_index(drop=True)

print("X_train_v8:", X_train_v8.shape)
print("X_test_v8:", X_test_v8.shape)
print("Target:", y.shape)
print("Categorical features from schema:", len(feature_schema.get("categorical_columns", feature_schema.get("categorical_features", []))))

display(y.describe())

X_train_v8: (8340, 73)
X_test_v8: (8341, 73)
Target: (8340,)
Categorical features from schema: 22


count    8.340000e+03
mean     3.712697e+04
std      3.679723e+04
min      9.000000e+02
25%      1.944075e+04
50%      2.962000e+04
75%      4.399000e+04
max      1.500000e+06
Name: Цена, dtype: float64

## 3. Общий holdout и вспомогательные функции

Для сопоставимого сравнения все компоненты обучаются на одинаковых строках train и оцениваются на одинаковом validation split. Стратификация по квантилям цены сохраняет в train и validation автомобили разных ценовых сегментов.

Метрика конкурса — MAPE, поэтому все оценки выводятся в процентах.

In [6]:
# ==============================================================
# 3. HOLDOUT AND COMMON HELPERS
# ============================================================== 

def mape_percent(y_true, y_pred) -> float:
    """MAPE в процентах; прогнозы ограничиваются положительными значениями."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1.0)
    return mean_absolute_percentage_error(y_true, y_pred) * 100


def make_stratification_bins(target: pd.Series, n_bins: int = 10) -> np.ndarray:
    """Квантили target для стратифицированного split."""
    return np.asarray(
        pd.qcut(
            target,
            q=n_bins,
            labels=False,
            duplicates="drop",
        )
    )


stratify_bins = make_stratification_bins(y)
all_positions = np.arange(len(y))

fit_indices, valid_indices = train_test_split(
    all_positions,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_bins,
)

fit_indices = np.sort(fit_indices)
valid_indices = np.sort(valid_indices)

assert len(set(fit_indices).intersection(valid_indices)) == 0
assert len(fit_indices) + len(valid_indices) == len(y)

print("Train rows:", len(fit_indices))
print("Validation rows:", len(valid_indices))
print("Train target median:", round(float(y.iloc[fit_indices].median()), 2))
print("Validation target median:", round(float(y.iloc[valid_indices].median()), 2))

Train rows: 6672
Validation rows: 1668
Train target median: 29570.0
Validation target median: 29673.5


## 4. Подготовка признаков для моделей

`CatBoost` работает с категориальными признаками напрямую. Линейному Ridge требуется one-hot encoding. Для text Ridge формируется отдельный текст: бренд, модель, полное название, двигатель, привод, коробка и тип кузова.

Такое разделение показывает, что компоненты ансамбля используют разные представления одного объекта.

In [7]:
# ==============================================================
# 4. FEATURE VIEWS FOR DIFFERENT MODELS
# ============================================================== 

categorical_v8 = feature_schema.get("categorical_columns", feature_schema.get("categorical_features", []))
missing_categorical = [
    column
    for column in categorical_v8
    if column not in X_train_v8.columns
]

if missing_categorical:
    raise KeyError(
        "В feature_schema есть колонки, которых нет в X_train_v8: "
        f"{missing_categorical}"
    )


def prepare_catboost_frame(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    """Единый формат категорий для CatBoost."""
    result = frame.copy()

    for column in categorical_columns:
        result[column] = (
            result[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return result


X_train_cb = prepare_catboost_frame(
    X_train_v8,
    categorical_v8,
)


def align_canonical_by_id(
    canonical_frame: pd.DataFrame,
    ids: pd.Series,
) -> pd.DataFrame:
    """Выравнивает raw canonical строки в порядке prepared dataset."""
    indexed = canonical_frame.copy()
    indexed[ID_COLUMN] = indexed[ID_COLUMN].astype(str)
    indexed = indexed.set_index(ID_COLUMN, drop=False)

    missing_ids = set(ids.astype(str)) - set(indexed.index)
    if missing_ids:
        raise KeyError(
            "Часть car_id отсутствует в canonical таблице: "
            f"{list(missing_ids)[:5]}"
        )

    return indexed.loc[ids.astype(str)].reset_index(drop=True)


train_canonical_aligned = align_canonical_by_id(
    train_canonical,
    train_ids,
)

TEXT_COLUMNS = [
    "Бренд",
    "Модель",
    "Полное название",
    "Двигатель",
    "Привод",
    "КПП",
    "Тип кузова",
]

missing_text_columns = [
    column
    for column in TEXT_COLUMNS
    if column not in train_canonical_aligned.columns
]

if missing_text_columns:
    raise KeyError(
        "В train_canonical не хватает текстовых колонок: "
        f"{missing_text_columns}"
    )


def make_vehicle_text(frame: pd.DataFrame) -> pd.Series:
    """Создаёт текстовое представление автомобиля для TF-IDF модели."""
    text_parts = []

    for column in TEXT_COLUMNS:
        text_parts.append(
            frame[column]
            .astype("string")
            .fillna("")
            .str.strip()
        )

    return pd.concat(text_parts, axis=1).agg(" ".join, axis=1)


vehicle_text = make_vehicle_text(train_canonical_aligned)

print("CatBoost V8 columns:", X_train_cb.shape[1])
print("Text examples:")
display(vehicle_text.head(3).to_frame("vehicle_text"))

CatBoost V8 columns: 73
Text examples:


,vehicle_text
0,SUBARU WRX 2018 SUBARU WRX STI SPEC R (AWD) 4 ...
1,SSANGYONG REXTON 2023 SSANGYONG REXTON ELX (AW...
2,NISSAN X-TRAIL 2011 NISSAN X-TRAIL ST (4X4) 4 ...


## 5. Ridge baseline

Ridge обучается на `log1p(Цена)`. Логарифм снижает влияние очень дорогих автомобилей и делает целевую переменную более близкой к симметричному распределению. После предсказания выполняется обратное преобразование `expm1`.

Это простой baseline, но он важен для ансамбля: линейная модель делает ошибки иначе, чем градиентный бустинг.

In [8]:
# ==============================================================
# 5. RIDGE BASELINE
# ============================================================== 

def make_one_hot_encoder() -> OneHotEncoder:
    """Совместимость со старыми и новыми версиями scikit-learn."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


numeric_v8 = [
    column
    for column in X_train_cb.columns
    if column not in categorical_v8
]

ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_v8,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", make_one_hot_encoder()),
                ]
            ),
            categorical_v8,
        ),
    ],
    remainder="drop",
)

ridge_model = Pipeline(
    steps=[
        ("preprocessor", ridge_preprocessor),
        ("model", Ridge(alpha=0.1, solver="lsqr")),
    ]
)

ridge_model.fit(
    X_train_cb.iloc[fit_indices],
    np.log1p(y.iloc[fit_indices]),
)

ridge_valid_pred = np.maximum(
    np.expm1(
        ridge_model.predict(
            X_train_cb.iloc[valid_indices]
        )
    ),
    1,
)

ridge_holdout_mape = mape_percent(
    y.iloc[valid_indices],
    ridge_valid_pred,
)

print("Ridge holdout MAPE:", round(ridge_holdout_mape, 6))

Ridge holdout MAPE: 13.817287


## 6. Text Ridge

Название автомобиля содержит редкие обозначения комплектаций и версий, которые могут быть неполно отражены в явных признаках. TF-IDF переводит текст в матрицу частот слов и биграмм, а Ridge строит линейный прогноз на этом представлении.

Text Ridge обычно слабее основной CatBoost-модели отдельно, но полезна в ансамбле за счёт другого источника информации.

In [9]:
# ==============================================================
# 6. TEXT RIDGE
# ============================================================== 

text_ridge_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=50_000,
                sublinear_tf=True,
            ),
        ),
        (
            "model",
            Ridge(alpha=1.0, solver="lsqr"),
        ),
    ]
)

text_ridge_model.fit(
    vehicle_text.iloc[fit_indices],
    np.log1p(y.iloc[fit_indices]),
)

text_valid_pred = np.maximum(
    np.expm1(
        text_ridge_model.predict(
            vehicle_text.iloc[valid_indices]
        )
    ),
    1,
)

text_holdout_mape = mape_percent(
    y.iloc[valid_indices],
    text_valid_pred,
)

print("Text Ridge holdout MAPE:", round(text_holdout_mape, 6))

Text Ridge holdout MAPE: 18.331346


## 7. CatBoost V8

V8 — основная нелинейная модель. Она использует технические параметры автомобиля и признаки, извлечённые из названия: нормализованные префиксы, комплектации, мощность, тип двигателя и флаги специальных версий.

CatBoost выбран потому, что естественно работает с категориальными признаками и умеет моделировать нелинейные взаимодействия, например «марка × год × пробег × двигатель».

In [10]:
# ==============================================================
# 7. CATBOOST V8
# ============================================================== 

CATBOOST_PARAMS = {
    "loss_function": "RMSE",
    "iterations": 3000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 5,
    "random_seed": RANDOM_STATE,
    "verbose": 500,
    "allow_writing_files": False,
}

v8_model = CatBoostRegressor(**CATBOOST_PARAMS)

v8_model.fit(
    X_train_cb.iloc[fit_indices],
    np.log1p(y.iloc[fit_indices]),
    cat_features=categorical_v8,
    eval_set=(
        X_train_cb.iloc[valid_indices],
        np.log1p(y.iloc[valid_indices]),
    ),
    early_stopping_rounds=200,
    use_best_model=True,
)

v8_valid_pred = np.maximum(
    np.expm1(
        v8_model.predict(
            X_train_cb.iloc[valid_indices]
        )
    ),
    1,
)

v8_holdout_mape = mape_percent(
    y.iloc[valid_indices],
    v8_valid_pred,
)

print("V8 holdout MAPE:", round(v8_holdout_mape, 6))

0:	learn: 0.6526009	test: 0.6564301	best: 0.6564301 (0)	total: 240ms	remaining: 11m 58s
500:	learn: 0.1544984	test: 0.2025570	best: 0.2025570 (500)	total: 51.1s	remaining: 4m 15s
1000:	learn: 0.1165359	test: 0.1930352	best: 0.1930352 (1000)	total: 1m 44s	remaining: 3m 27s
1500:	learn: 0.0945635	test: 0.1901886	best: 0.1901749 (1496)	total: 2m 38s	remaining: 2m 37s
2000:	learn: 0.0787487	test: 0.1880874	best: 0.1880673 (1993)	total: 3m 33s	remaining: 1m 46s
2500:	learn: 0.0664462	test: 0.1872844	best: 0.1872445 (2478)	total: 4m 29s	remaining: 53.7s
2999:	learn: 0.0571638	test: 0.1867817	best: 0.1867817 (2999)	total: 5m 22s	remaining: 0us

bestTest = 0.1867816902
bestIteration = 2999

V8 holdout MAPE: 12.330446


## 8. Fold-safe target statistics и CatBoost V10

Цена автомобиля сильно зависит от группы: марки, модели, года и версии. Поэтому добавляются сглаженные средние цены по таким группам.

Нельзя вычислять среднюю цену группы с использованием той же строки, для которой строится признак: это была бы утечка таргета. Поэтому в train target statistics строятся через внутренний cross-fitting, а для validation — только по внешнему train split.

In [11]:
# ==============================================================
# 8. FOLD-SAFE TARGET STATISTICS FOR V10
# ============================================================== 

GROUP_SPECS_V1 = {
    "brand_model": ["Бренд", "Модель"],
    "brand_model_year": ["Бренд", "Модель", "Год выпуска"],
    "title_prefix3": ["Название_префикс_3"],
    "title_prefix3_year": ["Название_префикс_3", "Год выпуска"],
    "title_normalized": ["Название_нормализованное_без_года"],
    "title_normalized_year": [
        "Название_нормализованное_без_года",
        "Год выпуска",
    ],
}

TARGET_STATS_SMOOTHING = 20.0
TARGET_STATS_N_SPLITS = 5

missing_group_columns = sorted(
    {
        column
        for columns in GROUP_SPECS_V1.values()
        for column in columns
        if column not in X_train_cb.columns
    }
)

if missing_group_columns:
    raise KeyError(
        "Для target statistics не хватает колонок: "
        f"{missing_group_columns}"
    )


def build_target_stats(
    X_reference: pd.DataFrame,
    y_reference: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float,
) -> pd.DataFrame:
    """Строит count и smoothed mean log-price на reference и применяет к X_apply."""
    if len(X_reference) != len(y_reference):
        raise ValueError("X_reference и y_reference имеют разный размер.")

    reference = X_reference.reset_index(drop=True).copy()
    apply = X_apply.reset_index(drop=True).copy()
    target_log = np.log1p(
        pd.Series(y_reference).reset_index(drop=True).astype(float)
    )

    global_log_mean = float(target_log.mean())
    stats_result = pd.DataFrame(index=apply.index)

    for feature_name, group_columns in group_specs.items():
        grouped = (
            reference[group_columns]
            .copy()
            .assign(_target_log=target_log)
            .groupby(group_columns, dropna=False)['_target_log']
            .agg(['count', 'mean'])
            .reset_index()
        )

        grouped[f"{feature_name}__log_mean"] = (
            (
                grouped['count'] * grouped['mean']
                + smoothing * global_log_mean
            )
            / (grouped['count'] + smoothing)
        )

        lookup = grouped[
            group_columns
            + [
                'count',
                f"{feature_name}__log_mean",
            ]
        ].rename(
            columns={
                'count': f"{feature_name}__count",
            }
        )

        merged = apply[group_columns].merge(
            lookup,
            on=group_columns,
            how='left',
            sort=False,
        )

        stats_result[f"{feature_name}__count"] = (
            merged[f"{feature_name}__count"]
            .fillna(0.0)
            .astype(float)
        )

        stats_result[f"{feature_name}__log_mean"] = (
            merged[f"{feature_name}__log_mean"]
            .fillna(global_log_mean)
            .astype(float)
        )

    return stats_result


def make_cross_fitted_target_stats(
    X_fit: pd.DataFrame,
    y_fit: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float = TARGET_STATS_SMOOTHING,
    n_splits: int = TARGET_STATS_N_SPLITS,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Создаёт cross-fitted stats для X_fit и full-reference stats для X_apply."""
    X_fit = X_fit.reset_index(drop=True)
    y_fit = pd.Series(y_fit).reset_index(drop=True)
    X_apply = X_apply.reset_index(drop=True)

    expected_columns = []
    for feature_name in group_specs:
        expected_columns.extend(
            [
                f"{feature_name}__count",
                f"{feature_name}__log_mean",
            ]
        )

    fit_stats = pd.DataFrame(
        index=X_fit.index,
        columns=expected_columns,
        dtype=float,
    )

    inner_bins = make_stratification_bins(y_fit)
    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    for inner_train_idx, inner_valid_idx in inner_cv.split(
        X_fit,
        inner_bins,
    ):
        fold_stats = build_target_stats(
            X_reference=X_fit.iloc[inner_train_idx],
            y_reference=y_fit.iloc[inner_train_idx],
            X_apply=X_fit.iloc[inner_valid_idx],
            group_specs=group_specs,
            smoothing=smoothing,
        )
        fit_stats.iloc[inner_valid_idx] = fold_stats.to_numpy()

    apply_stats = build_target_stats(
        X_reference=X_fit,
        y_reference=y_fit,
        X_apply=X_apply,
        group_specs=group_specs,
        smoothing=smoothing,
    )

    if fit_stats.isna().any().any():
        raise ValueError("Cross-fitted target statistics содержат NaN.")

    return fit_stats, apply_stats


X_fit_v8 = X_train_cb.iloc[fit_indices].reset_index(drop=True)
X_valid_v8 = X_train_cb.iloc[valid_indices].reset_index(drop=True)
y_fit_v10 = y.iloc[fit_indices].reset_index(drop=True)
y_valid_v10 = y.iloc[valid_indices].reset_index(drop=True)

X_fit_stats, X_valid_stats = make_cross_fitted_target_stats(
    X_fit=X_fit_v8,
    y_fit=y_fit_v10,
    X_apply=X_valid_v8,
    group_specs=GROUP_SPECS_V1,
)

X_fit_v10 = pd.concat(
    [X_fit_v8, X_fit_stats],
    axis=1,
)

X_valid_v10 = pd.concat(
    [X_valid_v8, X_valid_stats],
    axis=1,
)

assert list(X_fit_v10.columns) == list(X_valid_v10.columns)
assert X_fit_v10.shape[1] == X_train_cb.shape[1] + 12

print("V8 features:", X_train_cb.shape[1])
print("V10 features:", X_fit_v10.shape[1])
display(X_fit_stats.head(3))

V8 features: 73
V10 features: 85


,brand_model__count,brand_model__log_mean,brand_model_year__count,brand_model_year__log_mean,title_prefix3__count,title_prefix3__log_mean,title_prefix3_year__count,title_prefix3_year__log_mean,title_normalized__count,title_normalized__log_mean,title_normalized_year__count,title_normalized_year__log_mean
0,5.0,10.408000,4.0,10.389215,0.0,10.278391,0.0,10.278391,0.0,10.278391,0.0,10.278391
1,91.0,10.138756,2.0,10.207633,91.0,10.138756,2.0,10.207633,10.0,10.048469,1.0,10.236169
2,121.0,10.874560,9.0,10.687441,2.0,10.424497,1.0,10.360034,0.0,10.278905,0.0,10.278905


In [12]:
# ==============================================================
# 9. CATBOOST V10
# ============================================================== 

v10_model = CatBoostRegressor(**CATBOOST_PARAMS)

v10_model.fit(
    X_fit_v10,
    np.log1p(y_fit_v10),
    cat_features=categorical_v8,
    eval_set=(
        X_valid_v10,
        np.log1p(y_valid_v10),
    ),
    early_stopping_rounds=200,
    use_best_model=True,
)

v10_valid_pred = np.maximum(
    np.expm1(
        v10_model.predict(X_valid_v10)
    ),
    1,
)

v10_holdout_mape = mape_percent(
    y_valid_v10,
    v10_valid_pred,
)

print("V10 holdout MAPE:", round(v10_holdout_mape, 6))

0:	learn: 0.6527022	test: 0.6557685	best: 0.6557685 (0)	total: 42.5ms	remaining: 2m 7s
500:	learn: 0.1438139	test: 0.1972871	best: 0.1972871 (500)	total: 49s	remaining: 4m 4s
1000:	learn: 0.1041691	test: 0.1882293	best: 0.1882268 (999)	total: 1m 44s	remaining: 3m 29s
1500:	learn: 0.0815485	test: 0.1854197	best: 0.1854091 (1498)	total: 2m 38s	remaining: 2m 38s
2000:	learn: 0.0672963	test: 0.1841268	best: 0.1841103 (1998)	total: 3m 37s	remaining: 1m 48s
2500:	learn: 0.0560319	test: 0.1835252	best: 0.1834972 (2490)	total: 4m 41s	remaining: 56.2s
2999:	learn: 0.0472400	test: 0.1831251	best: 0.1831212 (2997)	total: 5m 44s	remaining: 0us

bestTest = 0.1831211903
bestIteration = 2997

Shrink model to first 2998 iterations.
V10 holdout MAPE: 12.134718


## 9. Retrieval K=3: ближайшие аналоги

Retrieval не обучает глобальную функцию цены. Вместо этого он ищет у обучающих объектов ближайшие аналоги автомобиля по марке, модели, году, пробегу, двигателю, коробке, приводу, кузову и комплектации.

Прогноз получается как взвешенное среднее логарифмов цен трёх ближайших аналогов. Этот компонент слабее CatBoost отдельно, но приносит в ансамбль локальную рыночную информацию.

In [15]:
# ==============================================================
# 10. RETRIEVAL K=3
# ============================================================== 

RETRIEVAL_NUMERIC_SPECS = {
    "Год выпуска": 3.0,
    "Пробег_log": 0.70,
    "Двигатель_объём_л": 0.70,
    "Двигатель_цилиндры": 1.5,
    "Двери_число": 2.0,
    "Кресла_число": 2.0,
}

RETRIEVAL_CATEGORICAL_WEIGHTS = {
    "Топливо": 0.35,
    "КПП": 0.25,
    "Привод": 0.25,
    "Тип кузова": 0.30,
    "Штат": 0.10,
    "Название_комплектация_1": 0.35,
    "Название_комплектация_2": 0.15,
}

RETRIEVAL_REQUIRED_COLUMNS = list(
    dict.fromkeys(
        [
            "Бренд",
            "Модель",
            "Пробег_число",
            *[
                column
                for column in RETRIEVAL_NUMERIC_SPECS
                if column != "Пробег_log"
            ],
            *RETRIEVAL_CATEGORICAL_WEIGHTS.keys(),
        ]
    )
)

    
missing_retrieval_columns = [
    column
    for column in RETRIEVAL_REQUIRED_COLUMNS
    if column not in X_train_cb.columns
]

if missing_retrieval_columns:
    raise KeyError(
        "Для retrieval не хватает признаков: "
        f"{missing_retrieval_columns}"
    )

RETRIEVAL_K = 3
DISTANCE_EPSILON = 0.15
MISSING_NUMERIC_PENALTY = 0.20
MISSING_CATEGORICAL_PENALTY = 0.15


def clean_retrieval_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .fillna("__MISSING__")
        .str.strip()
        .str.upper()
        .astype(str)
    )


def prepare_retrieval_frame(
    X: pd.DataFrame,
    target: pd.Series | None = None,
) -> pd.DataFrame:
    result = X[RETRIEVAL_REQUIRED_COLUMNS].copy().reset_index(drop=True)

    for column in [
        "Год выпуска",
        "Пробег_число",
        "Двигатель_объём_л",
        "Двигатель_цилиндры",
        "Двери_число",
        "Кресла_число",
    ]:
        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        ).astype(float)

    result["Пробег_log"] = np.log1p(
        result["Пробег_число"]
    )

    for column in RETRIEVAL_CATEGORICAL_WEIGHTS:
        result[column] = clean_retrieval_category(result[column])

    result["Бренд"] = clean_retrieval_category(result["Бренд"])
    result["Модель"] = clean_retrieval_category(result["Модель"])
    result["brand_model_key"] = (
        result["Бренд"] + "|||" + result["Модель"]
    )
    result["brand_key"] = result["Бренд"]

    if target is not None:
        result["target_log_price"] = np.log1p(
            pd.Series(target).reset_index(drop=True).astype(float)
        )

    return result


def comparable_distance(
    fit_block: pd.DataFrame,
    query_block: pd.DataFrame,
) -> np.ndarray:
    distance = np.zeros(
        (len(fit_block), len(query_block)),
        dtype=float,
    )

    for column, scale in RETRIEVAL_NUMERIC_SPECS.items():
        fit_values = fit_block[column].to_numpy(dtype=float)[:, None]
        query_values = query_block[column].to_numpy(dtype=float)[None, :]
        available = np.isfinite(fit_values) & np.isfinite(query_values)
        normalized_gap = np.abs(fit_values - query_values) / scale
        distance += np.where(
            available,
            normalized_gap ** 2,
            MISSING_NUMERIC_PENALTY,
        )

    for column, weight in RETRIEVAL_CATEGORICAL_WEIGHTS.items():
        fit_values = fit_block[column].to_numpy(dtype=object)[:, None]
        query_values = query_block[column].to_numpy(dtype=object)[None, :]
        available = (
            (fit_values != "__MISSING__")
            & (query_values != "__MISSING__")
        )
        mismatch = (fit_values != query_values).astype(float)
        distance += weight * np.where(
            available,
            mismatch,
            MISSING_CATEGORICAL_PENALTY,
        )

    return distance


def predict_retrieval_k3(
    fit_frame: pd.DataFrame,
    query_frame: pd.DataFrame,
) -> np.ndarray:
    """Возвращает price prediction по K ближайшим аналогам."""
    fit_frame = fit_frame.reset_index(drop=True)
    query_frame = query_frame.reset_index(drop=True)

    prediction_log = np.zeros(len(query_frame), dtype=float)

    brand_model_groups = fit_frame.groupby(
        "brand_model_key",
        sort=False,
    ).indices

    brand_groups = fit_frame.groupby(
        "brand_key",
        sort=False,
    ).indices

    for group_key, query_positions in query_frame.groupby(
        "brand_model_key",
        sort=False,
    ).indices.items():
        candidate_positions = brand_model_groups.get(group_key)

        if candidate_positions is None:
            brand_key = query_frame.loc[
                query_positions[0],
                "brand_key",
            ]
            candidate_positions = brand_groups.get(brand_key)

        if candidate_positions is None:
            candidate_positions = np.arange(len(fit_frame))

        fit_block = fit_frame.iloc[candidate_positions]
        query_block = query_frame.iloc[query_positions]

        distances = comparable_distance(fit_block, query_block)
        k_effective = min(RETRIEVAL_K, len(fit_block))

        nearest_positions = np.argpartition(
            distances,
            kth=k_effective - 1,
            axis=0,
        )[:k_effective]

        nearest_distances = np.take_along_axis(
            distances,
            nearest_positions,
            axis=0,
        )

        nearest_log_prices = fit_block[
            "target_log_price"
        ].to_numpy(dtype=float)[nearest_positions]

        weights = 1 / (
            nearest_distances + DISTANCE_EPSILON
        ) ** 2

        prediction_log[query_positions] = (
            weights * nearest_log_prices
        ).sum(axis=0) / weights.sum(axis=0)

    return np.maximum(np.expm1(prediction_log), 1)


retrieval_fit = prepare_retrieval_frame(
    X_train_cb.iloc[fit_indices],
    y.iloc[fit_indices],
)

retrieval_valid = prepare_retrieval_frame(
    X_train_cb.iloc[valid_indices],
)

retrieval_valid_pred = predict_retrieval_k3(
    retrieval_fit,
    retrieval_valid,
)

retrieval_holdout_mape = mape_percent(
    y.iloc[valid_indices],
    retrieval_valid_pred,
)

print("Retrieval K=3 holdout MAPE:", round(retrieval_holdout_mape, 6))

Retrieval K=3 holdout MAPE: 16.199569


## 10. Сравнение компонентов на holdout

Индивидуальное качество важно, но не является единственным критерием включения в ансамбль. Например, retrieval и text Ridge могут быть слабее CatBoost отдельно, однако их ошибки отличаются от ошибок основной модели — поэтому они улучшают итоговый ансамбль.

In [16]:
# ==============================================================
# 11. HOLDOUT COMPARISON
# ============================================================== 

holdout_comparison = pd.DataFrame(
    [
        {
            "component": "Ridge log-target",
            "holdout_mape_pct": ridge_holdout_mape,
            "role": "Линейный tabular baseline",
        },
        {
            "component": "Text Ridge TF-IDF",
            "holdout_mape_pct": text_holdout_mape,
            "role": "Сигнал из названия и текста",
        },
        {
            "component": "CatBoost V8",
            "holdout_mape_pct": v8_holdout_mape,
            "role": "Основная нелинейная модель",
        },
        {
            "component": "CatBoost V10",
            "holdout_mape_pct": v10_holdout_mape,
            "role": "V8 + fold-safe target statistics",
        },
        {
            "component": "Retrieval K=3",
            "holdout_mape_pct": retrieval_holdout_mape,
            "role": "Ближайшие аналоги",
        },
    ]
).sort_values("holdout_mape_pct")

HOLDOUT_COMPARISON_PATH = (
    REPORTS_DIR / "02_holdout_model_comparison.csv"
)

holdout_comparison.to_csv(
    HOLDOUT_COMPARISON_PATH,
    index=False,
)

print("Saved:", HOLDOUT_COMPARISON_PATH)
display(holdout_comparison.round(6))

Saved: C:\temp\shift_ml\reports\02_holdout_model_comparison.csv


,component,holdout_mape_pct,role
3,CatBoost V10,12.134718,V8 + fold-safe target statistics
2,CatBoost V8,12.330446,Основная нелинейная модель
0,Ridge log-target,13.817287,Линейный tabular baseline
4,Retrieval K=3,16.199569,Ближайшие аналоги
1,Text Ridge TF-IDF,18.331346,Сигнал из названия и текста


## 11. Зафиксированный выбор финального ансамбля

Финальные веса нельзя честно выбирать по одному holdout: это привело бы к подгонке. Поэтому они были выбраны ранее по 5-fold OOF-предсказаниям, а conditional calibration дополнительно проверялась nested validation.

Ниже фиксируется итоговая конфигурация, которая дала leaderboard score **12.66**. В этом notebook она не переоптимизируется.

In [17]:
# ==============================================================
# 12. FROZEN FINAL RECIPE FROM OOF RESEARCH
# ============================================================== 

FINAL_ENSEMBLE_RECIPE = {
    "leaderboard_score": 12.66,
    "submission_name": "submission_v10_conditional_disagree.zip",
    "ensemble_weights": {
        "v8": 0.348682,
        "ridge": 0.196562,
        "v10_stats_v8_pred": 0.161980,
        "retrieval_k3_pred": 0.132080,
        "stats": 0.101857,
        "text": 0.038848,
        "v5": 0.019991,
        "v6": 0.000000,
    },
    "disagreement_cut_points": [
        0.04946076,
        0.08297837,
    ],
    "disagreement_multipliers": {
        "low": 0.994,
        "medium": 0.980,
        "high": 0.951,
    },
    "nested_oof_mape_pct": {
        "raw_exact_v10_ensemble": 11.715950,
        "global_calibration": 11.572301,
        "conditional_disagreement_calibration": 11.472216,
    },
}

FINAL_RECIPE_PATH = (
    REPORTS_DIR / "02_final_ensemble_recipe_12_66.json"
)

with open(
    FINAL_RECIPE_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        FINAL_ENSEMBLE_RECIPE,
        file,
        ensure_ascii=False,
        indent=4,
    )

weights_table = pd.DataFrame(
    {
        "component": list(
            FINAL_ENSEMBLE_RECIPE[
                "ensemble_weights"
            ].keys()
        ),
        "weight": list(
            FINAL_ENSEMBLE_RECIPE[
                "ensemble_weights"
            ].values()
        ),
    }
).sort_values("weight", ascending=False)

nested_oof_table = pd.DataFrame(
    [
        {
            "stage": stage,
            "nested_oof_mape_pct": score,
        }
        for stage, score in FINAL_ENSEMBLE_RECIPE[
            "nested_oof_mape_pct"
        ].items()
    ]
)

print("Saved:", FINAL_RECIPE_PATH)

print("\nFinal ensemble weights")
display(weights_table.round(6))

print("\nNested OOF calibration effect")
display(nested_oof_table.round(6))

Saved: C:\temp\shift_ml\reports\02_final_ensemble_recipe_12_66.json

Final ensemble weights


,component,weight
0,v8,0.348682
1,ridge,0.196562
2,v10_stats_v8_pred,0.161980
3,retrieval_k3_pred,0.132080
4,stats,0.101857
5,text,0.038848
6,v5,0.019991
7,v6,0.000000



Nested OOF calibration effect


,stage,nested_oof_mape_pct
0,raw_exact_v10_ensemble,11.715950
1,global_calibration,11.572301
2,conditional_disagreement_calibration,11.472216


## 12. Conditional calibration по disagreement моделей

После получения прогнозов всех компонентов вычисляется относительный разброс их предсказаний:

```python
relative_std = std(component_predictions) / mean(component_predictions)
```

Большой разброс означает, что модели по-разному оценивают объект: например, автомобиль редкий, дорогой, имеет нестандартную комплектацию или плохо покрыт аналогами. В этом сегменте ансамбль чаще завышал цену, поэтому применялись три calibration-множителя:

| Уровень disagreement | Множитель |
|---|---:|
| Низкий | 0.994 |
| Средний | 0.980 |
| Высокий | 0.951 |

Nested OOF MAPE улучшился с `11.572301` после глобальной calibration до `11.472216` после conditional calibration. Именно эта схема была подтверждена финальным leaderboard score `12.66`.

## Выводы Notebook 2

- На holdout CatBoost V8 и V10 являются основными сильными компонентами.
- Ridge, text Ridge и retrieval дают альтернативные представления автомобиля: линейное, текстовое и локальное по ближайшим аналогам.
- Финальный прогноз строится ансамблем, потому что разнообразие ошибок важнее качества одной модели.
- Веса ансамбля и conditional calibration зафиксированы по OOF/nested validation, а не подбираются на одном holdout.
- Неудачные направления — direct MAPE CatBoost, residual retrieval и smooth calibration — не включены в финальный pipeline, потому что не дали устойчивого улучшения.